<a href="https://colab.research.google.com/github/Sunidhishree/flyrank-ml-internship1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess, getpass

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sunidhishree/flyrank-ml-internship1"
REPO_DIR = "flyrank-ml-internship1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

get_ipython().system('pip -q install duckdb huggingface_hub')

# Use Colab's Secrets panel (key icon 🔑) to store HF_TOKEN — never paste it in a cell
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Ready.")

Ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
verify_grain = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n_days
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1
    ORDER BY n_days DESC
    LIMIT 5
""").df()
verify_grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,n_days
0,content_b7e512995f79d5a6,31
1,content_05597932fe4da067,31
2,content_905aa32a0230694e,31
3,content_05434271b257bb68,31
4,content_d056587ff7faca0c,31


One row = one content page (content_hash_id), aggregated over a single mid-panel month, 2026-03. I'm using a monthly aggregation rather than daily rows because my lane (clustering) groups pages by their overall behavior pattern for a period, not day-by-day fluctuation. I deliberately picked 2026-03 instead of the _sample table (June 2026, the final/outcome month) to avoid building on the one month that's the natural label-outcome window for any past→future prediction task.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (knowable at decision time): gsc_impressions, gsc_clicks, gsc_avg_position — summed/averaged per content item within the month.
Label: none — clustering is unsupervised, so there is no predicted label. (For the leakage-trap demo in Section 3, I'll construct a temporary synthetic label purely to demonstrate the effect, then discard it.)
Context (not a feature, but useful for interpretation): client_hash_id — used only for grouping/filtering, never as a model input, since it's a pseudonymous ID.
Excluded: trend_direction and trend_pct (or any column describing month-over-month change) — these describe an outcome pattern rather than a behavioral input, and including them would leak decline-related information into what should be a purely descriptive clustering of current behavior.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Should be empty — confirms true grain is one row per date x client x content:")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Should be empty — confirms true grain is one row per date x client x content:


,report_date,client_hash_id,content_hash_id,n


In [5]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT content_hash_id) AS n_content, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
span

,n_rows,min_d,max_d,n_content,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [6]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows_available
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_rows_available
0,9841378,413966.0


In [7]:
features = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month,
           AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr_month,
           COUNT(*) AS days_reported
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1
    HAVING impressions_month >= 50
""").df()
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions_month,clicks_month,avg_position_month,avg_ctr_month,days_reported
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.004334,31
1,content_05597932fe4da067,57.0,0.0,2.714744,0.000000,31
2,content_905aa32a0230694e,149.0,0.0,6.481453,0.000000,31
3,content_05434271b257bb68,1421.0,6.0,6.320337,0.004077,31
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,0.003589,31


impressions_month — knowable at decision time: direct sum of already-logged data.
clicks_month — same, directly logged.
avg_position_month — average of daily observed rankings, known once month closes.
avg_ctr_month — derived only from same-window clicks/impressions.
days_reported — structural count, known immediately.

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

demo = features.copy()
demo['is_low_ctr'] = (demo['avg_ctr_month'] < demo['avg_ctr_month'].median()).astype(int)

honest_cols = ['impressions_month', 'avg_position_month', 'days_reported']
X_honest = demo[honest_cols].fillna(0)
y = demo['is_low_ctr']

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.25, random_state=42)
m1 = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("Honest AUC:", round(roc_auc_score(yte, m1.predict_proba(Xte)[:,1]), 3))

leaky_cols = honest_cols + ['clicks_month']
X_leak = demo[leaky_cols].fillna(0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leak, y, test_size=0.25, random_state=42)
m2 = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2)
print("Leaky AUC (clicks_month included, entangled with the label):",
      round(roc_auc_score(yte2, m2.predict_proba(Xte2)[:,1]), 3))

del X_leak, leaky_cols, m2
print("Leak removed. Keeping only the honest feature set going forward.")

Honest AUC: 0.746
Leaky AUC (clicks_month included, entangled with the label): 0.988
Leak removed. Keeping only the honest feature set going forward.


Adding clicks_month — mathematically part of how the label was defined — inflated AUC sharply toward-perfect. This reproduces the notebook 02 leakage lesson on real warehouse data: a feature entangled with the label teaches the model to reconstruct the label, not to find genuine signal. Removed and kept only the honest features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me why a page behaves a certain way — only what the observed pattern is. Specific limits of this slice: (1) client history depth is unbalanced — some clients have far more months of data than others, so a single-month clustering may underweight newer clients; (2) rows before a client's ga4_data_start have GA4 columns zero-filled, not truly "no engagement," so any GA4-based feature must filter on the availability flag; (3) this is a single month's snapshot — cluster assignments may shift month to month, and I have not yet tested stability across multiple months.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.